# 02 — Data Cleaning

Ce notebook applique les règles de nettoyage établies dans `01_data_understanding.ipynb`. Chaque étape rappelle la décision et sa justification avant le code.

## 1. Chargement des données brutes

In [42]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/logements-sociaux-finances-a-paris.csv", sep=';')
print("Shape initiale :", df.shape)
df.head()


Shape initiale : (4174, 19)


,id_livraison,adresse_programme,code_postal,ville,annee,bs,nb_logmt_total,nb_plai,nb_plus,nb_pluscd,nb_pls,mode_real,commentaires,arrdt,nature_programme,coord_x_l93,coord_y_l93,geo_shape,geo_point_2d
0,2000011,16-20 RUE DES MEUNIERS,75012,Paris,2001,RES.URB.,62,0,62,0,0,acquisition conventionnement,NaN,12,logement familial,655823.1529,6.859495e+06,"{""coordinates"": [2.398175442412243, 48.8339981...","48.83399817959342, 2.398175442412243"
1,2000035,25 RUE DES ANNELETS,75019,Paris,2001,RIVP,18,10,8,0,0,acquisition réhabilitation,NaN,19,logement familial,655225.8330,6.864433e+06,"{""coordinates"": [2.3895183752327043, 48.878361...","48.8783619998301, 2.3895183752327043"
2,2000044,LOT D1B - 86 RUE DES HAIES,75020,Paris,2001,RIVP,6,0,6,0,0,acquisition réhabilitation,NaN,20,logement familial,656258.4385,6.861828e+06,"{""coordinates"": [2.403865260493333, 48.8550036...","48.85500366971155, 2.403865260493333"
3,2000048,31 QUAI DE VALMY,75010,Paris,2001,PARIS HABITAT,28,0,22,0,6,acquisition conventionnement,NaN,10,logement familial,653551.4260,6.863384e+06,"{""coordinates"": [2.366804047083707, 48.8688074...","48.86880747974437, 2.366804047083707"
4,2000062,19 RUE DES PLANTES,75014,Paris,2001,PARIS HABITAT,38,0,15,0,23,acquisition conventionnement,NaN,14,logement familial,650356.4410,6.859151e+06,"{""coordinates"": [2.3237456731730535, 48.830499...","48.83049983027532, 2.3237456731730535"


In [43]:
df_o=df.copy()

## 2. Suppression des colonnes sans valeur analytique

- `commentaires` : 100% de valeurs manquantes (0 valeur non-nulle) → aucune information.
- `ville` : constante ("Paris" partout) → aucune valeur discriminante.


In [44]:
cols_to_drop = ['commentaires', 'ville']
df = df.drop(columns=cols_to_drop)
print(f"Colonnes supprimées : {cols_to_drop}")
print("Shape après suppression :", df.shape)


Colonnes supprimées : ['commentaires', 'ville']
Shape après suppression : (4174, 17)


## 3. Exclusion des lignes avec valeurs négatives

2 lignes concernées (`id_livraison` 2000587 et 2000900). Le total (`nb_logmt_total`) est cohérent
avec la somme des sous-catégories dans les deux cas → probablement des corrections/annulations
d'une déclaration antérieure, pas des lignes de logements réelles. Impact négligeable (2/4174 lignes).


In [45]:
neg_cols = ['nb_logmt_total', 'nb_plai', 'nb_plus', 'nb_pluscd', 'nb_pls']
neg_mask = (df[neg_cols] < 0).any(axis=1)

print(f"Lignes exclues : {neg_mask.sum()}")
print(df.loc[neg_mask, ['id_livraison', 'adresse_programme', 'annee'] + neg_cols])

df = df.loc[~neg_mask].copy()
print("\nShape après exclusion :", df.shape)


Lignes exclues : 2
     id_livraison                              adresse_programme  annee  \
2528      2000587  56-58, RUE LEON FROT 3, RUE CARRIERE MAINGUET   2010   
3457      2000900       16-22, RUE FERNAND LEGER (FOYER MURIERS)   2010   

      nb_logmt_total  nb_plai  nb_plus  nb_pluscd  nb_pls  
2528               5        6       -1          0       0  
3457              -8       -8        0          0       0  

Shape après exclusion : (4172, 17)


## 4. `id_livraison` dupliqués — conservés tels quels

Les 32 IDs dupliqués (65 lignes) représentent des tranches de financement différentes ou des phases
de livraison distinctes d'un même programme — **ce ne sont pas des doublons**. `id_livraison` identifie
un programme, pas une ligne unique. Aucune suppression, mais on documente le comportement pour ne pas
utiliser `id_livraison` comme clé unique plus tard dans l'analyse.


In [46]:
n_dupe_ids = df['id_livraison'].duplicated(keep=False).sum()
print(f"Lignes concernées par un id_livraison dupliqué (conservées) : {n_dupe_ids}")
print("Rappel : id_livraison n'est PAS une clé unique par ligne dans ce dataset.")


Lignes concernées par un id_livraison dupliqué (conservées) : 61
Rappel : id_livraison n'est PAS une clé unique par ligne dans ce dataset.


## 5. `bs` = "(vide)" → converti en vraie valeur manquante (NaN)

C'est un placeholder texte pour une valeur manquante, pas une vraie catégorie. Le convertir en `NaN`
évite qu'il soit compté comme une modalité à part entière dans de futures agrégations (`value_counts`,
`groupby`, etc.), même si la colonne n'est pas utilisée dans l'hypothèse principale.


In [47]:
n_vide = (df['bs'] == '(vide)').sum()
df['bs'] = df['bs'].replace('(vide)', np.nan)
print(f"'(vide)' remplacé par NaN sur {n_vide} lignes")


'(vide)' remplacé par NaN sur 109 lignes


## 6. Correction des types de données

- `annee` : déjà en `int64`, OK.
- `mode_real`, `nature_programme` : conversion en `category` pour optimiser la mémoire et
  rendre les valeurs autorisées explicites.


In [48]:
cat_cols = ['mode_real', 'nature_programme']
for c in cat_cols:
    df[c] = df[c].astype('category')


df.dtypes


id_livraison              str
adresse_programme         str
code_postal               str
annee                   int64
bs                        str
nb_logmt_total          int64
nb_plai                 int64
nb_plus                 int64
nb_pluscd               int64
nb_pls                  int64
mode_real            category
arrdt                   int64
nature_programme     category
coord_x_l93           float64
coord_y_l93           float64
geo_shape                 str
geo_point_2d              str
dtype: object

## 7. Vérification nb_logmt_total / somme des sous-catégories
## (diagnostic établi dans 01_data_understanding.ipynb  0 écart trouvé)

Confirmé : aucune action de nettoyage nécessaire, les 4 colonnes de financement 
sont parfaitement cohérentes avec le total sur 100% des lignes.

In [49]:
df['sum_categories'] = df[['nb_plai', 'nb_plus', 'nb_pluscd', 'nb_pls']].sum(axis=1)
n_mismatch = (df['sum_categories'] != df['nb_logmt_total']).sum()
print(f"Écarts trouvés : {n_mismatch}")
assert n_mismatch == 0, "Incohérence détectée — vérifier avant de continuer"
df = df.drop(columns=['sum_categories'])

Écarts trouvés : 0


## 8. Vérification id_livraison / mode_real
## (diagnostic établi dans 01_data_understanding.ipynb)

7 id_livraison présentent un mode_real incohérent entre leurs lignes sur le brut. 
Après exclusion des 2 lignes négatives (section 3), ce nombre attendu est de 5. 
6 des 7 cas sont des tranches légitimes du même programme (adresse similaire, à 
des variations d'écriture près) ; 1 cas (2002196) est une réutilisation probable 
d'identifiant sur deux adresses distinctes. Comme l'analyse se fait au niveau 
ligne, ceci n'affecte pas l'agrégation. Aucune action de nettoyage nécessaire.

In [50]:
n_incoherent = (df.groupby('id_livraison')['mode_real'].nunique() > 1).sum()
print(f"id_livraison incohérents après nettoyage : {n_incoherent}")
assert n_incoherent == 5, "Le nombre attendu (5) ne correspond pas — vérifier"

id_livraison incohérents après nettoyage : 5


## 9. Vérifications finales


In [59]:
print("Shape finale :", df.shape)
print("\nValeurs manquantes restantes :")
print(df.isnull().sum()[df.isnull().sum() > 0])
print("\nValeurs négatives restantes :")
print((df[neg_cols] < 0).sum().sum())
print("\nLignes dupliquées exactes :", df.duplicated().sum())
print("\nNombre des lignes supprimés  :",len(df_o)-len(df))


Shape finale : (4172, 17)

Valeurs manquantes restantes :
bs    109
dtype: int64

Valeurs négatives restantes :
0

Lignes dupliquées exactes : 0

Nombre des lignes supprimés  : 2


## 10. Sauvegarde du dataset nettoyé

In [52]:

import os
os.makedirs("../data/processed", exist_ok=True)
df.to_csv("../data/processed/logements-sociaux-paris-clean.csv", index=False, sep=';')
print("Sauvegardé :", df.shape)


Sauvegardé : (4172, 17)
